<a href="https://colab.research.google.com/github/ANUPAM7209/mlp/blob/main/ML_Preprocessing_Clinic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🩺 ML Preprocessing Clinic: Diagnose the Bug

You've already learned these concepts. This notebook is a **self-test**, not a lecture.

**How each round works:**
1. Read the scenario / code snippet.
2. **Predict** what will happen — jot your answer.
3. Run the **reveal cell** to see the actual result.
4. Open the **explanation** (click the ▶ arrow) to check your reasoning.

No peeking at the explanation before you predict — that's the whole point. Let's go.

Run the setup cell below first (it just loads libraries and a synthetic dataset).

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Synthetic Titanic-like dataset (self-contained — no external file needed).
# Missingness rates mirror the real Titanic dataset: ~20% Age missing, ~77% Cabin missing.
np.random.seed(42)
titanic = pd.DataFrame({
    "PassengerId": range(1, 892),
    "Survived": np.random.randint(0, 2, 891),
    "Pclass": np.random.choice([1, 2, 3], 891),
    "Age": np.where(np.random.rand(891) < 0.20, np.nan,
                     np.random.normal(29, 14, 891).round(1)),
    "Cabin": np.where(np.random.rand(891) < 0.77, np.nan,
                       "C" + pd.Series(np.random.randint(1, 200, 891)).astype(str)),
    "Embarked": np.random.choice(["S", "C", "Q", np.nan], 891, p=[0.70, 0.15, 0.14, 0.01]),
})

print("Setup complete. Available: titanic (DataFrame)")

Setup complete. Available: titanic (DataFrame)


## Exercise 1: Unpacking `load_iris()`

```python
from sklearn.datasets import load_iris
X, y = load_iris()
```

**Predict:** What happens when this runs?

- **A）** Works fine — X is the feature matrix, y is the labels
- **B）** Raises an error
- **C）** X and y both come back empty
- **D）** Returns strings instead of numbers

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.datasets import load_iris
try:
    X, y = load_iris()
    print("Ran fine. X =", str(X)[:50])
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

❌ ValueError: too many values to unpack (expected 2)


In [ ]:
data = load_iris()

<details>
<summary>💡 Click to reveal the explanation</summary>

`load_iris()` by default returns a `Bunch` — a dict-like container with `.data`, `.target`, `.feature_names`, etc. — not a 2-tuple. Trying to unpack a Bunch into two variables fails because it doesn't iterate as `(X, y)`.
**Fix:**
```python
X, y = load_iris(return_X_y=True)
```

</details>

## Exercise 2:  The Missing Value

```python
csv = "age,city\n25,A\n?,B\n30,A\n"
d = pd.read_csv(io.StringIO(csv))
d["age"].mean()
```

**Predict:** What happens when this runs?

- **A）** Returns 27.5
- **B）** Returns NaN
- **C）** Raises a TypeError
- **D）** Returns the literal string "?"

Write your guess below, then run the reveal cell.

In [ ]:
import io
csv = "age,city\n25,A\n?,B\n30,A\n"
d = pd.read_csv(io.StringIO(csv))
print("dtype without na_values:", d["age"].dtype)
try:
    print(d["age"].mean())
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

d2 = pd.read_csv(io.StringIO(csv), na_values="?")
print("\ndtype with na_values='?':", d2["age"].dtype, "| mean:", d2["age"].mean())

<details>
<summary>💡 Click to reveal the explanation</summary>

pandas only auto-recognizes a fixed default list of missing-value tokens (`NA`, `NaN`, empty string, etc). `"?"` isn't on that list, so the whole `age` column is read as **strings**, and `.mean()` fails outright with a TypeError.

**Fix:**
```python
pd.read_csv(io.StringIO(csv), na_values="?")
```

</details>

## Exercise 3:  The `-999` Sentinel

```python
age = pd.Series([25, 30, 28, -999, 35, -999])
age.mean()
```

**Predict:** What happens when this runs?

- **A）** ≈ 29.5, since -999 is just ignored
- **B）** A wildly negative number, far from the real average
- **C）** Raises a ValueError
- **D）** NaN

Write your guess below, then run the reveal cell.

In [ ]:
age = pd.Series([25, 30, 28, -999, 35, -999])
naive = age.mean()
real = age.replace(-999, np.nan).mean()
print("naive mean:", round(naive, 1))
print("real mean :", round(real, 1))

<details>
<summary>💡 Click to reveal the explanation</summary>

`-999` is a numeric value, so pandas happily averages it in — no error, no warning, just a badly wrong number. This is more dangerous than a crash because **nothing tells you something is wrong.** Sentinel values must be replaced with NaN explicitly before any aggregation.

**Fix:**
```python
age.replace(-999, np.nan).mean()
```

</details>

## Exercise 4:  `dropna()`




```python
titanic.dropna()               # vs
titanic.dropna(subset=['Age'])
```

**Predict:** What happens when this runs?

- **A）** Both keep roughly the same number of rows
- **B）** Plain dropna() keeps almost nothing; subset=['Age'] keeps most rows
- **C）** Plain dropna() keeps more rows than the subset version
- **D）** Both raise an error since Cabin has so many NaNs

Write your guess below, then run the reveal cell.

In [ ]:
print("rows before:               ", len(titanic))
print("after dropna():             ", len(titanic.dropna()))
print("after dropna(subset=['Age']):", len(titanic.dropna(subset=["Age"])))

<details>
<summary>💡 Click to reveal the explanation</summary>

`dropna()` with no arguments drops a row if **any** column has a NaN. Since `Cabin` is missing on the large majority of rows, a blanket `dropna()` wipes out nearly the whole dataset — even though `Age` (the column you probably care about) is fine on most of those rows. Always scope `dropna()` with `subset=[...]` unless you truly mean 'drop if anything at all is missing.'

**Fix:**
```python
titanic.dropna(subset=['Age'])
```

</details>

## Exercise 5:  Mean-Imputing

```python
from sklearn.impute import SimpleImputer
Xc = np.array([["a"],["b"],[np.nan]], dtype=object)
SimpleImputer(strategy="mean").fit_transform(Xc)
```

**Predict:** What happens when this runs?

- **A）** Works fine, fills with the 'average' letter
- **B）** Raises a ValueError
- **C）** Fills with 0
- **D）** Silently drops the NaN row

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.impute import SimpleImputer
Xc = np.array([["a"], ["b"], [np.nan]], dtype=object)
try:
    SimpleImputer(strategy="mean").fit_transform(Xc)
    print("ran (unexpected)")
except Exception as e:
    print(f"❌ {type(e).__name__}: {str(e)[:70]}")

fixed = SimpleImputer(strategy="most_frequent").fit_transform(Xc)
print("fix with most_frequent:", fixed.ravel())

<details>
<summary>💡 Click to reveal the explanation</summary>

`strategy='mean'` needs numbers to average — you can't take the mean of the letters 'a' and 'b'. For categorical/text columns use `strategy='most_frequent'` or `strategy='constant'` with a fill value.

**Fix:**
```python
SimpleImputer(strategy="most_frequent")
```

</details>

## Exercise 6:  Scaling *Before* Imputing

```python
from sklearn.preprocessing import StandardScaler
Xn = np.array([[1.],[2.],[np.nan],[4.]])
StandardScaler().fit_transform(Xn)
```

**Predict:** What happens when this runs?

- **A）** The NaN gets filled in automatically by the scaler
- **B）** The output still has a NaN in the same spot
- **C）** Raises an error immediately
- **D）** The NaN becomes 0

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.preprocessing import StandardScaler
Xn = np.array([[1.], [2.], [np.nan], [4.]])
out = StandardScaler().fit_transform(Xn)
print("scaled output:", out.ravel())

<details>
<summary>💡 Click to reveal the explanation</summary>

`StandardScaler` only rescales values — it does **not** impute. Any NaN going in comes out as NaN (arithmetic with NaN just produces NaN), and it will silently propagate into your model. The rule of thumb: **impute first, then scale** (this is exactly what an sklearn `Pipeline` with imputer → scaler enforces for you).

</details>

## Exercise 7: Evaluating on Train vs. Test

```python
dt = DecisionTreeClassifier().fit(Xtr, ytr)
accuracy_score(ytr, dt.predict(Xtr))   # vs
accuracy_score(yte, dt.predict(Xte))
```

**Predict:** What happens when this runs?

- **A）** Train and test accuracy will be about the same
- **B）** Train accuracy will be much higher than test accuracy
- **C）** Test accuracy will be much higher than train accuracy
- **D）** Both will be exactly 1.0

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

Xb, yb = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.3, random_state=0)

dt = DecisionTreeClassifier(random_state=0).fit(Xtr, ytr)
print("accuracy on TRAIN:", round(accuracy_score(ytr, dt.predict(Xtr)), 3))
print("accuracy on TEST: ", round(accuracy_score(yte, dt.predict(Xte)), 3))

<details>
<summary>💡 Click to reveal the explanation</summary>

An unconstrained decision tree can memorize its training data, so it scores 100% on rows it has already seen — that number tells you nothing about generalization. The test-set score (data the model never trained on) is the honest estimate of real-world performance. Always evaluate on held-out data.

</details>

## Exercise 8: `LabelEncoder` on a Feature Matrix



```python
from sklearn.preprocessing import LabelEncoder
LabelEncoder().fit_transform(np.array([["a","b"],["c","d"]]))
```

**Predict:** What happens when this runs?

- **A）** Works fine, encodes each column separately
- **B）** Raises a ValueError
- **C）** Returns the array unchanged
- **D）** Encodes only the first column

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.preprocessing import LabelEncoder
try:
    LabelEncoder().fit_transform(np.array([["a", "b"], ["c", "d"]]))
    print("ran")
except Exception as e:
    print(f"❌ {type(e).__name__}: {str(e)[:70]}")

<details>
<summary>💡 Click to reveal the explanation</summary>

`LabelEncoder` is built for a single 1D array of **labels/targets** (e.g. your `y`), not a 2D matrix of **features**. Feeding it a 2D array raises an error. For encoding multiple feature columns, use `OrdinalEncoder` or `OneHotEncoder` instead.

</details>

## Exercise 9: OneHotEncoder Meets an Unseen Category

```python
ohe = OneHotEncoder().fit([["A"],["B"]])
ohe.transform([["C"]])   # "C" was never seen during fit
```

**Predict:** What happens when this runs?

- **A）** Works fine, treats C as a new category
- **B）** Raises a ValueError
- **C）** Encodes C as all-zeros with no error
- **D）** Encodes C as the average of A and B

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
ohe.fit(np.array([["A"], ["B"]]))
try:
    ohe.transform(np.array([["C"]]))
    print("ran")
except Exception as e:
    print(f"❌ {type(e).__name__}: {str(e)[:70]}")

ohe2 = OneHotEncoder(handle_unknown="ignore").fit(np.array([["A"], ["B"]]))
print("fix with handle_unknown='ignore':", ohe2.transform(np.array([["C"]])).toarray())

<details>
<summary>💡 Click to reveal the explanation</summary>

By default `OneHotEncoder` raises if it sees a category at transform time that it never saw during `fit` — this happens all the time when your test set (or production data) contains a rare category your training set didn't. `handle_unknown='ignore'` encodes unseen categories as all-zeros instead of crashing.

**Fix:**
```python
OneHotEncoder(handle_unknown="ignore")
```

</details>

## Exercise 10: `pd.get_dummies` Train/Test Mismatch

```python
tr = pd.DataFrame({"port": ["S","C","Q"]})
te = pd.DataFrame({"port": ["S","S"]})
pd.get_dummies(tr).columns   # vs
pd.get_dummies(te).columns
```

**Predict:** What happens when this runs?

- **A）** Same columns in both — get_dummies is consistent
- **B）** Test set ends up with fewer columns than train
- **C）** Both raise an error
- **D）** Test set ends up with more columns than train

Write your guess below, then run the reveal cell.

In [ ]:
tr = pd.DataFrame({"port": ["S", "C", "Q"]})
te = pd.DataFrame({"port": ["S", "S"]})
print("train columns:", list(pd.get_dummies(tr).columns))
print("test columns: ", list(pd.get_dummies(te).columns))

<details>
<summary>💡 Click to reveal the explanation</summary>

`get_dummies` only creates a column for categories that are actually **present** in the data you pass it. If the test set happens not to contain 'C' or 'Q', those columns simply don't exist there — so a model trained on the train-set columns will break (or silently misalign) on test data with a different column set. Fit an encoder like `OneHotEncoder` on the training data, then `.transform()` the test data with it (never fit-and-encode train/test separately).

</details>

## Exercise 11: Target Leakage

```python
full = np.column_stack([Xb, yb])   # oops — last column IS the target
LogisticRegression().fit(Xtr2, ytr2).score(Xte2, yte2)
```

**Predict:** What happens when this runs?

- **A）** Accuracy looks normal, around 90-95%
- **B）** Accuracy looks suspiciously close to 100%
- **C）** Raises an error since the target is duplicated
- **D）** Accuracy drops to near 50% (random guessing)

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.linear_model import LogisticRegression

full = np.column_stack([Xb, yb])   # target column accidentally included as a feature
Xtr2, Xte2, ytr2, yte2 = train_test_split(full, yb, test_size=0.3, random_state=0)
lr = LogisticRegression(max_iter=200).fit(Xtr2, ytr2)
print("accuracy with target leaked into features:", round(accuracy_score(yte2, lr.predict(Xte2)), 3))

<details>
<summary>💡 Click to reveal the explanation</summary>

When the target column (or something that directly encodes it) ends up inside X, the model essentially just 'looks up the answer.' Accuracy jumps to a suspicious ~99%+. **Any time performance looks too good to be true, check for leakage first** — it usually is.

</details>

## Exercise 12: A Pipeline With No Final Estimator

```python
p = Pipeline([("s", StandardScaler())])
p.fit(Xtr, ytr)
p.predict(Xte)
```

**Predict:** What happens when this runs?

- **A）** Works fine, predicts using the scaled values
- **B）** Raises an AttributeError
- **C）** Returns the scaled Xte unchanged
- **D）** Silently returns all zeros

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.pipeline import Pipeline
p = Pipeline([("s", StandardScaler())])
try:
    p.fit(Xtr, ytr)
    p.predict(Xte)
    print("ran")
except Exception as e:
    print(f"❌ {type(e).__name__}: {str(e)[:70]}")

<details>
<summary>💡 Click to reveal the explanation</summary>

A `Pipeline`'s final step determines what methods it exposes. A pipeline that ends in a **transformer** (like `StandardScaler`) can `.transform()` data but has no `.predict()` — there's no estimator in it. You need a classifier/regressor as the last step for `.predict()` to exist.

</details>

## Exercise 13: Feature Selection Before Train/Test Split

```python
Xk = SelectKBest(f_classif, k=5).fit_transform(Xb, yb)   # fit on ALL of X, y — before any split
```

**Predict:** What happens when this runs?

- **A）** Raises an error since y includes test labels
- **B）** Runs fine with no error — but it's a subtle leakage bug
- **C）** Automatically excludes the last 30% as a held-out test set
- **D）** Only works if Xb and yb are pre-split

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
Xk = SelectKBest(f_classif, k=5).fit_transform(Xb, yb)
print("ran fine, shape:", Xk.shape, "-- but it used ALL of y, including future test rows")

<details>
<summary>💡 Click to reveal the explanation</summary>

This runs without any error, which is exactly what makes it dangerous. Selecting features using the **full** dataset (including rows that will later become your test set) means your 'test' evaluation is no longer truly independent — the feature selection step has already seen those labels. Feature selection should happen **inside** the training fold only (e.g. inside a `Pipeline`, so it's redone correctly for every cross-validation split).

</details>

## Exercise 14: Using a Regression Score Function for Classification

```python
SelectKBest(f_regression, k=5).fit_transform(Xb, yb)   # yb is a classification target (0/1)
```

**Predict:** What happens when this runs?

- **A）** Raises a clear error telling you f_regression is wrong here
- **B）** Runs with no error, but the feature ranking is quietly wrong
- **C）** Automatically falls back to f_classif
- **D）** Returns identical results to f_classif

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
try:
    Xk2 = SelectKBest(f_regression, k=5).fit_transform(Xb, yb)
    print("ran (no error!) shape:", Xk2.shape, "-> silently wrong")
except Exception as e:
    print(f"❌ {type(e).__name__}: {e}")

<details>
<summary>💡 Click to reveal the explanation</summary>

scikit-learn does **not** check that your scoring function matches your problem type. `f_regression` measures linear correlation with a continuous target; `f_classif` (ANOVA F-value) is the right one for a categorical/classification target. Using the wrong one won't crash — it'll just quietly rank the wrong features as 'best.'

**Fix:**
```python
SelectKBest(f_classif, k=5)
```

</details>

## Exercise 15: PCA Without Scaling

```python
PCA(n_components=3).fit(Xb).explained_variance_ratio_          # vs
PCA(n_components=3).fit(StandardScaler().fit_transform(Xb)).explained_variance_ratio_
```

**Predict:** What happens when this runs?

- **A）** Nearly identical explained-variance ratios either way
- **B）** Without scaling, PC1 dominates (~98%); with scaling it's much more balanced
- **C）** Without scaling, PCA raises an error
- **D）** Scaling has no effect on PCA results

Write your guess below, then run the reveal cell.

In [ ]:
from sklearn.decomposition import PCA

p_noscale = PCA(n_components=3).fit(Xb)
p_scaled = PCA(n_components=3).fit(StandardScaler().fit_transform(Xb))

print("no-scale explained var ratio:", p_noscale.explained_variance_ratio_.round(3))
print("scaled   explained var ratio:", p_scaled.explained_variance_ratio_.round(3))

<details>
<summary>💡 Click to reveal the explanation</summary>

PCA finds directions of maximum **variance** — and variance is scale-dependent. If one feature happens to be measured on a much larger numeric scale than the others, it will dominate the first principal component even if it's not actually the most 'important' feature statistically. Standardizing features first (mean 0, unit variance) puts every feature on equal footing before PCA looks for structure.

</details>

## 🏁 Self-Score

Count how many you predicted correctly out of 15.

| # | Pitfall | One-line fix |
|---|---|---|
| 1 | `Bunch` unpacking | `return_X_y=True` |
| 2 | `"?"` not recognized as NaN | `na_values="?"` |
| 3 | Sentinel value (`-999`) skews stats | `.replace(-999, np.nan)` before aggregating |
| 4 | `dropna()` drops too much | scope with `subset=[...]` |
| 5 | Mean-imputing text | `strategy="most_frequent"` |
| 6 | Scaling before imputing | impute, **then** scale |
| 7 | Evaluating on train data | always score on held-out test data |
| 8 | `LabelEncoder` on features | use `OrdinalEncoder`/`OneHotEncoder` for X |
| 9 | OHE unseen category | `handle_unknown="ignore"` |
| 10 | `get_dummies` train/test mismatch | fit an encoder on train, transform test |
| 11 | Target leaked into features | audit every feature for target info |
| 12 | Pipeline with no estimator | end the pipeline with a classifier/regressor |
| 13 | Feature selection before split | select inside the training fold only |
| 14 | Wrong score function | `f_classif` for classification, `f_regression` for regression |
| 15 | PCA without scaling | `StandardScaler` before `PCA` |

**12-15 correct:** You've internalized these — nice work.
**7-11 correct:** Solid, but review the ones you missed before your next project.
**Below 7:** Worth re-running this notebook in a week — these bugs are sneaky precisely because they don't announce themselves.